# Batch Model Monitoring Demo - Train & Log

Trains a customer churn prediction model (XGBoost binary classifier) using:
- CUSTOMERS table (demographics + usage features)
- SUBSCRIPTION_EVENTS table (behavioral signals)

Logs to the Snowflake Model Registry with `task=TABULAR_BINARY_CLASSIFICATION`
(required for model monitoring).

In [ ]:
!pip install --upgrade "snowflake-ml-python>=1.7.1" snowflake-connector-python xgboost scikit-learn

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.use_schema("ML_DEMOS.BATCH_MONITORING")
session.use_warehouse("ML_DEMO_WH")
print("Connected.")

In [ ]:
# Build training dataset with synthetic churn labels
# Churn probability is driven by: short tenure, high charges, many tickets, low usage, month-to-month
import pandas as pd
import numpy as np

customers_df = session.table("CUSTOMERS").to_pandas()

np.random.seed(42)

def compute_churn_probability(row):
    p = 0.15  # base churn rate
    if row["TENURE_MONTHS"] < 6:
        p += 0.20
    elif row["TENURE_MONTHS"] < 12:
        p += 0.10
    if row["CONTRACT_TYPE"] == "MONTH_TO_MONTH":
        p += 0.15
    if row["NUM_SUPPORT_TICKETS"] >= 3:
        p += 0.15
    elif row["NUM_SUPPORT_TICKETS"] >= 1:
        p += 0.05
    if row["DAYS_SINCE_LAST_LOGIN"] >= 14:
        p += 0.15
    elif row["DAYS_SINCE_LAST_LOGIN"] >= 7:
        p += 0.08
    if row["AVG_MONTHLY_USAGE_HOURS"] < 20:
        p += 0.12
    if row["MONTHLY_CHARGES"] > 80:
        p += 0.08
    return min(p, 0.95)

customers_df["CHURN_PROB"] = customers_df.apply(compute_churn_probability, axis=1)
customers_df["CHURNED"] = (np.random.random(len(customers_df)) < customers_df["CHURN_PROB"]).astype(int)

print(f"Dataset shape: {customers_df.shape}")
print(f"\nChurn distribution:")
print(customers_df["CHURNED"].value_counts())
print(f"\nChurn rate: {customers_df['CHURNED'].mean():.1%}")

In [ ]:
# Prepare features for XGBoost
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

NUMERIC_FEATURES = [
    "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
    "NUM_SUPPORT_TICKETS", "DAYS_SINCE_LAST_LOGIN", "AVG_MONTHLY_USAGE_HOURS", "AGE",
]
CATEGORICAL_FEATURES = ["CONTRACT_TYPE", "PLAN_TYPE"]
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = customers_df[FEATURE_COLUMNS]
y = customers_df["CHURNED"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_transformer = SimpleImputer(strategy="constant", fill_value=0)
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, NUMERIC_FEATURES),
    ("cat", categorical_transformer, CATEGORICAL_FEATURES),
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1,
        objective="binary:logistic", random_state=42,
        eval_metric="logloss",
    )),
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["retained", "churned"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
# Log model to Snowflake Model Registry
from snowflake.ml.registry import Registry
from snowflake.ml.model import task

registry = Registry(session=session, database_name="ML_DEMOS", schema_name="BATCH_MONITORING")

mv = registry.log_model(
    pipeline,
    model_name="CHURN_PREDICTOR",
    version_name="V1",
    sample_input_data=X_train.head(5),
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
    comment="XGBoost churn classifier for batch monitoring demo",
    metrics={"auc_roc": roc_auc_score(y_test, y_proba)},
)
print(f"Model logged: {mv.model_name} / {mv.version_name}")

In [ ]:
# Verify model in registry
models_df = registry.show_models()
print(models_df[models_df["name"] == "CHURN_PREDICTOR"][["name", "versions", "comment"]])